# Read the customer validation results

After training, this notebook reads the saved outputs and explains what each tells us. The reference result in `docs/reference_results.csv` comes from the supplied LightGBM notebook. Results read here belong to the selected local experiment.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display
from amex_default.config import load_config, resolve

ROOT = Path.cwd()
if not (ROOT / "configs").is_dir():
    ROOT = ROOT.parent
cfg = load_config(ROOT / "configs" / "baseline.json")
output = resolve(cfg, "output")

## Pooled OOF metrics

AMEX, ROC-AUC, and average precision increase with better ranking. Log loss and Brier score decrease with better probability forecasts. Mean prediction and observed default rate help interpret overall calibration.

In [ ]:
metrics = json.loads((output / "metrics.json").read_text())
display(pd.Series(metrics["pooled_oof"], name="value").to_frame())
print("Mean fold AMEX:", metrics["mean_fold_amex"])

## Variation across folds

Compare fold scores and best iterations. This shows how much the fitted result changes across customer subsets.

In [ ]:
display(pd.read_csv(output / "fold_metrics.csv"))

## Out-of-fold predictions

Every row contains a customer, its target, a fold ID, and its OOF prediction. The predictions are useful for analysing score distributions and calibration in the sampled population.

In [ ]:
oof = pd.read_parquet(output / "oof.parquet")
display(oof.head())
display(oof.groupby("target")["prediction"].describe())

## Curves and feature importance

Generate charts with `amex report`. Feature importance reports mean split gain across folds and describes the model's predictive use of each feature.

In [ ]:
for name in ["discrimination.png", "calibration_and_scores.png", "feature_importance.png"]:
    display(Image(filename=str(output / "figures" / name)))
display(pd.read_csv(output / "feature_importance.csv").head(20))

## Compare reference and experiment

The reference scores use the Quickstart's own feature set, data representation, and fold setup. Read them as context alongside the local experiment settings saved in `run.json`.

In [ ]:
reference = pd.read_csv(ROOT / "docs" / "reference_results.csv")
display(reference)
manifest = json.loads((output / "run.json").read_text())
print(json.dumps({"config": manifest["config"], "versions": manifest["versions"]}, indent=2))

## Inspect feature availability

After `amex profile`, the following table ranks train/test differences in feature missingness. Begin with the largest absolute differences and compare their importance in the model.

In [ ]:
missing = pd.read_csv(output / "eda" / "feature_missingness.csv")
if "test_minus_train" in missing:
    missing = missing.assign(absolute_difference=missing["test_minus_train"].abs())
    missing = missing.sort_values("absolute_difference", ascending=False)
display(missing.head(20))